# 24 — Semisupervised Master Dataset Design

## Objetivo

Rediseñar formalmente la integración de los datasets públicos existentes para un enfoque de **aprendizaje semisupervisado**.

Este notebook no entrena todavía ningún modelo y tampoco concatena filas incompatibles. Su propósito es definir:

- qué fuentes pueden aportar datos etiquetados `L`;
- qué fuentes pueden aportar datos no etiquetados `U`;
- qué fuentes quedan solo como proxy o referencia;
- el esquema del dataset maestro;
- las reglas de procedencia de etiquetas;
- las reglas de integración que deberán cumplirse antes de construir `L ∪ U`;
- los artefactos que deberá consumir el notebook 25.

La idea metodológica será:

$$
D = L \cup U
$$

donde `L` contiene instancias con X + Y Nielsen válida y `U` contiene instancias con X compatibles pero Y desconocida.

Para el modelo Nielsen principal, `NaN` significa **no etiquetado**, nunca ausencia de violación.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_DIR = PROJECT_DIR / "data" / "semisupervised"
SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
SEMISUP_INTERIM_DIR = SEMISUP_DIR / "interim"
SEMISUP_MASTER_DIR = SEMISUP_DIR / "master"

for directory in [
    SEMISUP_DIR,
    SEMISUP_REPORTS_DIR,
    SEMISUP_INTERIM_DIR,
    SEMISUP_MASTER_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SEMISUP_DIR:", SEMISUP_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning
SEMISUP_DIR: /Users/uriel/Documents/machine_learning/data/semisupervised


## 2. Definir targets

In [2]:
TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

HEURISTIC_NAMES = {
    "H1_violation": "Visibility of system status",
    "H4_violation": "Consistency and standards",
    "H5_violation": "Error prevention",
    "H6_violation": "Recognition rather than recall"
}

print(TARGETS)


['H1_violation', 'H4_violation', 'H5_violation', 'H6_violation']


## 3. Principios metodológicos de integración

Las reglas de esta nueva fase son:

1. No realizar unión horizontal entre datasets sin una clave real compartida.
2. No concatenar verticalmente hasta definir un espacio X empírico compatible.
3. No convertir ratings, smells o proxies en etiquetas Nielsen directas.
4. `NaN` en H1/H4/H5/H6 significa *desconocido/no evaluado*.
5. Las pseudo-etiquetas futuras deben conservar procedencia distinta de las etiquetas observadas.
6. El dataset maestro debe conservar `source_dataset`, `source_instance_id`, modalidad y procedencia.
7. `source_dataset` será metadata de trazabilidad, no predictor del modelo.
8. Una fila puede ser `L` para una heurística y `U` para otra.
9. Cada target se modelará inicialmente como un problema binario independiente.
10. El conjunto etiquetado de evaluación nunca podrá contaminarse con pseudo-etiquetas.


## 4. Registro metodológico de fuentes

In [3]:
source_registry = pd.DataFrame([
    {
        "source_dataset": "Catching_UX_Flaws",
        "known_unit": "website",
        "feature_modality": "STATIC_WEB_SOURCE",
        "x_evidence": "Static web structural proxies recovered for 2 confirmed repositories.",
        "y_evidence": "Explicit Nielsen LLM labels H1/H4/H5/H6.",
        "same_instance_xy": True,
        "direct_nielsen_y": True,
        "label_level": "LLM_DIRECT",
        "semisupervised_candidate_role": "L_CANDIDATE",
        "known_limitation": "Only 2 strongly confirmed sites; all recovered targets are positive.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "29042675",
        "known_unit": "prototype",
        "feature_modality": "FIGMA_BEHANCE_SHAREPOINT_ARTIFACT",
        "x_evidence": "No canonical web X recovered.",
        "y_evidence": "Direct Nielsen LLM evidence recovered from evaluation documents.",
        "same_instance_xy": "PARTIAL",
        "direct_nielsen_y": True,
        "label_level": "LLM_DIRECT_CANDIDATE",
        "semisupervised_candidate_role": "L_CANDIDATE_PENDING_X",
        "known_limitation": "15 prototypes with candidate targets; recovered direct labels are positive-only and canonical web X is absent.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "UICrit_RICO",
        "known_unit": "android_ui",
        "feature_modality": "ANDROID_UI_HIERARCHY",
        "x_evidence": "1000 same-interface structural feature rows.",
        "y_evidence": "Broad usability/aesthetics/learnability/efficiency ratings.",
        "same_instance_xy": True,
        "direct_nielsen_y": False,
        "label_level": "RATING_PROXY",
        "semisupervised_candidate_role": "U_CANDIDATE_FOR_NIELSEN",
        "known_limitation": "Android structural variables are construct proxies, not exact web canonical metrics.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "Dashboards",
        "known_unit": "behavior_session_or_log",
        "feature_modality": "WEB_BEHAVIOR_LOG",
        "x_evidence": "Behavioral logs and derived auxiliary X.",
        "y_evidence": "No reproducible join from raw logs to P+DB+Q outcome rows.",
        "same_instance_xy": False,
        "direct_nielsen_y": False,
        "label_level": "NONE",
        "semisupervised_candidate_role": "U_CANDIDATE",
        "known_limitation": "No valid supervised join; only X-side use is defensible.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "A11y_CUA",
        "known_unit": "task_execution",
        "feature_modality": "MIXED_WEB_DESKTOP_INTERACTION",
        "x_evidence": "Behavior/event-derived features from exploratory notebooks.",
        "y_evidence": "No direct Nielsen H1/H4/H5/H6 labels.",
        "same_instance_xy": False,
        "direct_nielsen_y": False,
        "label_level": "NONE",
        "semisupervised_candidate_role": "U_CANDIDATE_PENDING_FEATURE_AUDIT",
        "known_limitation": "Contains web and non-web tasks; modality-derived features must remain NaN when unavailable.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "P3M",
        "known_unit": "screenshot",
        "feature_modality": "VISUAL_SCREENSHOT",
        "x_evidence": "Visual artifacts.",
        "y_evidence": "GPT-4 Nielsen positive issue rows.",
        "same_instance_xy": "PARTIAL",
        "direct_nielsen_y": True,
        "label_level": "LLM_DIRECT_POSITIVE_ONLY",
        "semisupervised_candidate_role": "LABEL_REFERENCE",
        "known_limitation": "Visual modality mismatch and negative-label semantics are absent.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "PHR",
        "known_unit": "portal_task_or_evaluation",
        "feature_modality": "PORTAL_EVALUATION",
        "x_evidence": "Limited for current canonical feature space.",
        "y_evidence": "Human adapted heuristic labels H01-H08.",
        "same_instance_xy": "UNKNOWN",
        "direct_nielsen_y": False,
        "label_level": "HUMAN_PROXY",
        "semisupervised_candidate_role": "LABEL_REFERENCE",
        "known_limitation": "Dataset-specific H01-H08 must not be mapped positionally to Nielsen H1-H8.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "SERENE",
        "known_unit": "issue_or_website",
        "feature_modality": "USABILITY_EVALUATION",
        "x_evidence": "Insufficient compatible X for current model.",
        "y_evidence": "Human expert general usability ground truth; no Nielsen heuristic column.",
        "same_instance_xy": "PARTIAL",
        "direct_nielsen_y": False,
        "label_level": "HUMAN_PROXY",
        "semisupervised_candidate_role": "LABEL_REFERENCE",
        "known_limitation": "Cannot derive H1/H4/H5/H6 directly.",
        "include_in_master_design": True
    },
    {
        "source_dataset": "AMUSED",
        "known_unit": "participant_interaction",
        "feature_modality": "WEB_INTERACTION",
        "x_evidence": "Potentially rich interaction logs.",
        "y_evidence": "Usability smells; Nielsen mappings would be proxies.",
        "same_instance_xy": "PENDING_ACCESS",
        "direct_nielsen_y": False,
        "label_level": "PENDING_ACCESS",
        "semisupervised_candidate_role": "PENDING_ACCESS",
        "known_limitation": "Restricted dataset access not yet granted.",
        "include_in_master_design": False
    }
])

display(source_registry)


,source_dataset,known_unit,feature_modality,x_evidence,y_evidence,same_instance_xy,direct_nielsen_y,label_level,semisupervised_candidate_role,known_limitation,include_in_master_design
0,Catching_UX_Flaws,website,STATIC_WEB_SOURCE,Static web structural proxies recovered for 2 confirmed repositories.,Explicit Nielsen LLM labels H1/H4/H5/H6.,True,True,LLM_DIRECT,L_CANDIDATE,Only 2 strongly confirmed sites; all recovered targets are positive.,True
1,29042675,prototype,FIGMA_BEHANCE_SHAREPOINT_ARTIFACT,No canonical web X recovered.,Direct Nielsen LLM evidence recovered from evaluation documents.,PARTIAL,True,LLM_DIRECT_CANDIDATE,L_CANDIDATE_PENDING_X,15 prototypes with candidate targets; recovered direct labels are positive-only and canonical web X is absent.,True
2,UICrit_RICO,android_ui,ANDROID_UI_HIERARCHY,1000 same-interface structural feature rows.,Broad usability/aesthetics/learnability/efficiency ratings.,True,False,RATING_PROXY,U_CANDIDATE_FOR_NIELSEN,"Android structural variables are construct proxies, not exact web canonical metrics.",True
3,Dashboards,behavior_session_or_log,WEB_BEHAVIOR_LOG,Behavioral logs and derived auxiliary X.,No reproducible join from raw logs to P+DB+Q outcome rows.,False,False,NONE,U_CANDIDATE,No valid supervised join; only X-side use is defensible.,True
4,A11y_CUA,task_execution,MIXED_WEB_DESKTOP_INTERACTION,Behavior/event-derived features from exploratory notebooks.,No direct Nielsen H1/H4/H5/H6 labels.,False,False,NONE,U_CANDIDATE_PENDING_FEATURE_AUDIT,Contains web and non-web tasks; modality-derived features must remain NaN when unavailable.,True
5,P3M,screenshot,VISUAL_SCREENSHOT,Visual artifacts.,GPT-4 Nielsen positive issue rows.,PARTIAL,True,LLM_DIRECT_POSITIVE_ONLY,LABEL_REFERENCE,Visual modality mismatch and negative-label semantics are absent.,True
6,PHR,portal_task_or_evaluation,PORTAL_EVALUATION,Limited for current canonical feature space.,Human adapted heuristic labels H01-H08.,UNKNOWN,False,HUMAN_PROXY,LABEL_REFERENCE,Dataset-specific H01-H08 must not be mapped positionally to Nielsen H1-H8.,True
7,SERENE,issue_or_website,USABILITY_EVALUATION,Insufficient compatible X for current model.,Human expert general usability ground truth; no Nielsen heuristic column.,PARTIAL,False,HUMAN_PROXY,LABEL_REFERENCE,Cannot derive H1/H4/H5/H6 directly.,True
8,AMUSED,participant_interaction,WEB_INTERACTION,Potentially rich interaction logs.,Usability smells; Nielsen mappings would be proxies.,PENDING_ACCESS,False,PENDING_ACCESS,PENDING_ACCESS,Restricted dataset access not yet granted.,False


## 5. Diferencia entre rol de fuente y rol por target

No se asignará una única columna global `labeled/unlabeled` para toda la fila.

Ejemplo:

```text
instance X:
H1 = 1
H4 = NaN
H5 = 0
H6 = NaN
```

La misma instancia es `L` para H1 y H5, pero `U` para H4 y H6.

Por eso el rol final se derivará por target en el notebook 26.


## 6. Niveles de procedencia de etiquetas

In [4]:
label_provenance_registry = pd.DataFrame([
    {
        "label_level": "HUMAN_DIRECT",
        "usable_as_observed_Y": True,
        "usable_as_primary_test": True,
        "pseudo_label": False,
        "description": "Etiqueta Nielsen explícita emitida por evaluador humano."
    },
    {
        "label_level": "LLM_DIRECT",
        "usable_as_observed_Y": True,
        "usable_as_primary_test": True,
        "pseudo_label": False,
        "description": "Etiqueta Nielsen explícita emitida por LLM; debe mantenerse separada de HUMAN_DIRECT."
    },
    {
        "label_level": "LLM_DIRECT_CANDIDATE",
        "usable_as_observed_Y": "AFTER_SOURCE_SPECIFIC_VALIDATION",
        "usable_as_primary_test": False,
        "pseudo_label": False,
        "description": "Evidencia directa recuperada, pero con limitaciones de extracción/mapeo o cobertura."
    },
    {
        "label_level": "HUMAN_PROXY",
        "usable_as_observed_Y": False,
        "usable_as_primary_test": False,
        "pseudo_label": False,
        "description": "Etiqueta humana relacionada, pero no Nielsen directa."
    },
    {
        "label_level": "RATING_PROXY",
        "usable_as_observed_Y": False,
        "usable_as_primary_test": False,
        "pseudo_label": False,
        "description": "Rating general usado solo en experimentos proxy."
    },
    {
        "label_level": "EXPERIMENTAL_PROXY",
        "usable_as_observed_Y": False,
        "usable_as_primary_test": False,
        "pseudo_label": False,
        "description": "Proxy experimental derivado de una variable relacionada."
    },
    {
        "label_level": "PSEUDO_LABEL",
        "usable_as_observed_Y": False,
        "usable_as_primary_test": False,
        "pseudo_label": True,
        "description": "Etiqueta generada por el algoritmo semisupervisado; nunca ground truth."
    },
    {
        "label_level": "NONE",
        "usable_as_observed_Y": False,
        "usable_as_primary_test": False,
        "pseudo_label": False,
        "description": "Instancia sin etiqueta Nielsen."
    }
])

display(label_provenance_registry)


,label_level,usable_as_observed_Y,usable_as_primary_test,pseudo_label,description
0,HUMAN_DIRECT,True,True,False,Etiqueta Nielsen explícita emitida por evaluador humano.
1,LLM_DIRECT,True,True,False,Etiqueta Nielsen explícita emitida por LLM; debe mantenerse separada de HUMAN_DIRECT.
2,LLM_DIRECT_CANDIDATE,AFTER_SOURCE_SPECIFIC_VALIDATION,False,False,"Evidencia directa recuperada, pero con limitaciones de extracción/mapeo o cobertura."
3,HUMAN_PROXY,False,False,False,"Etiqueta humana relacionada, pero no Nielsen directa."
4,RATING_PROXY,False,False,False,Rating general usado solo en experimentos proxy.
5,EXPERIMENTAL_PROXY,False,False,False,Proxy experimental derivado de una variable relacionada.
6,PSEUDO_LABEL,False,False,True,Etiqueta generada por el algoritmo semisupervisado; nunca ground truth.
7,NONE,False,False,False,Instancia sin etiqueta Nielsen.


## 7. Esquema del dataset maestro

In [5]:
master_schema_rows = [
    {"column": "master_instance_id", "role": "ID", "required": True, "description": "ID global único construido a partir de source_dataset + source_instance_id."},
    {"column": "source_dataset", "role": "PROVENANCE", "required": True, "description": "Dataset público de origen; no se utilizará como predictor."},
    {"column": "source_instance_id", "role": "PROVENANCE", "required": True, "description": "ID nativo o reproducible dentro de la fuente."},
    {"column": "source_unit", "role": "PROVENANCE", "required": True, "description": "Unidad original: website, prototype, task execution, Android UI, etc."},
    {"column": "feature_modality", "role": "PROVENANCE", "required": True, "description": "Modalidad de X."},
    {"column": "platform", "role": "CONTEXT", "required": False, "description": "Sitio/aplicación/plataforma cuando sea observable."},
    {"column": "task_type", "role": "CONTEXT", "required": False, "description": "Tipo de tarea cuando exista."},
    {"column": "page_type", "role": "CONTEXT", "required": False, "description": "Tipo de página/interfaz cuando pueda definirse reproduciblemente."},
    {"column": "<X_common>", "role": "X", "required": "DEFINED_IN_NOTEBOOK_25", "description": "Espacio común empírico; no se fija todavía."},
    {"column": "H1_violation", "role": "Y", "required": False, "description": "0/1 si existe etiqueta observada válida; NaN si no fue evaluada."},
    {"column": "H4_violation", "role": "Y", "required": False, "description": "0/1 si existe etiqueta observada válida; NaN si no fue evaluada."},
    {"column": "H5_violation", "role": "Y", "required": False, "description": "0/1 si existe etiqueta observada válida; NaN si no fue evaluada."},
    {"column": "H6_violation", "role": "Y", "required": False, "description": "0/1 si existe etiqueta observada válida; NaN si no fue evaluada."},
    {"column": "label_source", "role": "LABEL_PROVENANCE", "required": False, "description": "Identidad/tipo de anotador o mecanismo de generación."},
    {"column": "label_level", "role": "LABEL_PROVENANCE", "required": True, "description": "HUMAN_DIRECT, LLM_DIRECT, proxy, NONE, etc."},
    {"column": "label_confidence", "role": "LABEL_PROVENANCE", "required": False, "description": "Confianza si la fuente la proporciona; no inventarla."},
    {"column": "is_pseudo_label", "role": "LABEL_PROVENANCE", "required": True, "description": "True solo para etiquetas generadas por SSL."},
    {"column": "group_id", "role": "SPLIT", "required": True, "description": "Grupo para prevenir leakage: sitio/prototipo/aplicación según fuente."}
]

master_schema = pd.DataFrame(master_schema_rows)
display(master_schema)


,column,role,required,description
0,master_instance_id,ID,True,ID global único construido a partir de source_dataset + source_instance_id.
1,source_dataset,PROVENANCE,True,Dataset público de origen; no se utilizará como predictor.
2,source_instance_id,PROVENANCE,True,ID nativo o reproducible dentro de la fuente.
3,source_unit,PROVENANCE,True,"Unidad original: website, prototype, task execution, Android UI, etc."
4,feature_modality,PROVENANCE,True,Modalidad de X.
5,platform,CONTEXT,False,Sitio/aplicación/plataforma cuando sea observable.
6,task_type,CONTEXT,False,Tipo de tarea cuando exista.
7,page_type,CONTEXT,False,Tipo de página/interfaz cuando pueda definirse reproduciblemente.
8,<X_common>,X,DEFINED_IN_NOTEBOOK_25,Espacio común empírico; no se fija todavía.
9,H1_violation,Y,False,0/1 si existe etiqueta observada válida; NaN si no fue evaluada.


## 8. Regla de ID global

La identidad final será:

```text
master_instance_id =
    normalize(source_dataset)
    + "__"
    + normalize(source_instance_id)
```

Nunca se utilizará el índice de concatenación de pandas como identidad científica de la fila.


In [6]:
def normalize_id_part(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def make_master_instance_id(source_dataset, source_instance_id):
    return (
        normalize_id_part(source_dataset)
        + "__"
        + normalize_id_part(source_instance_id)
    )


example_id = make_master_instance_id(
    "Catching_UX_Flaws",
    "startbootstrap-sb-admin-2"
)

print(example_id)


catching_ux_flaws__startbootstrap_sb_admin_2


## 9. Reglas de integración

In [7]:
integration_rules = pd.DataFrame([
    {"rule_id": "R01", "rule": "NO_ROW_WISE_HORIZONTAL_JOIN_WITHOUT_KEY", "mandatory": True, "description": "No unir datasets diferentes por orden, índice o similitud nominal sin clave reproducible."},
    {"rule_id": "R02", "rule": "VERTICAL_UNION_ONLY_AFTER_FEATURE_HARMONIZATION", "mandatory": True, "description": "No concatenar L y U hasta que notebook 25 defina X_common."},
    {"rule_id": "R03", "rule": "UNKNOWN_LABEL_IS_NAN", "mandatory": True, "description": "No rellenar Y faltante con 0."},
    {"rule_id": "R04", "rule": "PROXY_NOT_EQUAL_DIRECT", "mandatory": True, "description": "RATING_PROXY/HUMAN_PROXY/EXPERIMENTAL_PROXY no se convierten a Nielsen directa."},
    {"rule_id": "R05", "rule": "PSEUDO_LABEL_PROVENANCE_REQUIRED", "mandatory": True, "description": "Toda pseudo-etiqueta debe marcarse explícitamente y quedar fuera del ground truth."},
    {"rule_id": "R06", "rule": "SOURCE_DATASET_NOT_PREDICTOR", "mandatory": True, "description": "La fuente solo se conserva para trazabilidad, agrupamiento y análisis."},
    {"rule_id": "R07", "rule": "GROUPED_SPLIT", "mandatory": True, "description": "El mismo sitio/prototipo/aplicación no puede aparecer en train y test."},
    {"rule_id": "R08", "rule": "TARGET_SPECIFIC_L_U", "mandatory": True, "description": "L y U se construyen separadamente para H1/H4/H5/H6."},
    {"rule_id": "R09", "rule": "TEST_NEVER_PSEUDO_LABELED", "mandatory": True, "description": "El conjunto de evaluación etiquetado permanece aislado durante SSL."},
    {"rule_id": "R10", "rule": "NO_GLOBAL_ZERO_FILL", "mandatory": True, "description": "Los faltantes de X conservan su semántica; no se reemplazan globalmente por cero."}
])

display(integration_rules)


,rule_id,rule,mandatory,description
0,R01,NO_ROW_WISE_HORIZONTAL_JOIN_WITHOUT_KEY,True,"No unir datasets diferentes por orden, índice o similitud nominal sin clave reproducible."
1,R02,VERTICAL_UNION_ONLY_AFTER_FEATURE_HARMONIZATION,True,No concatenar L y U hasta que notebook 25 defina X_common.
2,R03,UNKNOWN_LABEL_IS_NAN,True,No rellenar Y faltante con 0.
3,R04,PROXY_NOT_EQUAL_DIRECT,True,RATING_PROXY/HUMAN_PROXY/EXPERIMENTAL_PROXY no se convierten a Nielsen directa.
4,R05,PSEUDO_LABEL_PROVENANCE_REQUIRED,True,Toda pseudo-etiqueta debe marcarse explícitamente y quedar fuera del ground truth.
5,R06,SOURCE_DATASET_NOT_PREDICTOR,True,"La fuente solo se conserva para trazabilidad, agrupamiento y análisis."
6,R07,GROUPED_SPLIT,True,El mismo sitio/prototipo/aplicación no puede aparecer en train y test.
7,R08,TARGET_SPECIFIC_L_U,True,L y U se construyen separadamente para H1/H4/H5/H6.
8,R09,TEST_NEVER_PSEUDO_LABELED,True,El conjunto de evaluación etiquetado permanece aislado durante SSL.
9,R10,NO_GLOBAL_ZERO_FILL,True,Los faltantes de X conservan su semántica; no se reemplazan globalmente por cero.


## 10. Inventario automático de artefactos ya generados

Se inspeccionan únicamente nombres y rutas. Este paso no decide compatibilidad semántica.


In [8]:
candidate_dirs = [
    PUBLIC_INTERIM_DIR,
    REPORTS_DIR
]

patterns = [
    "*uicrit*",
    "*rico*",
    "*catching*",
    "*dashboard*",
    "*a11y*",
    "*29042675*",
    "*vt_*",
    "*feature*compatib*",
    "*source*role*",
    "*training*strategy*"
]

artifact_records = []
seen = set()

for directory in candidate_dirs:
    if not directory.exists():
        continue

    for pattern in patterns:
        for path in directory.rglob(pattern):
            if (
                not path.is_file()
                or path.suffix.lower() not in {".csv", ".json", ".parquet"}
            ):
                continue

            key = str(path.resolve())
            if key in seen:
                continue

            seen.add(key)

            artifact_records.append({
                "name": path.name,
                "suffix": path.suffix.lower(),
                "path": str(path.relative_to(PROJECT_DIR)),
                "size_bytes": path.stat().st_size
            })

artifact_inventory = pd.DataFrame(artifact_records)

if not artifact_inventory.empty:
    artifact_inventory = (
        artifact_inventory
        .sort_values(["name", "path"])
        .reset_index(drop=True)
    )

print(
    "Artefactos candidatos encontrados:",
    len(artifact_inventory)
)

display(artifact_inventory)


Artefactos candidatos encontrados: 71


,name,suffix,path,size_bytes
0,catching_confirmed_extension_summary.csv,.csv,reports/tables/catching_confirmed_extension_summary.csv,882
1,catching_confirmed_html_inventory.csv,.csv,reports/tables/catching_confirmed_html_inventory.csv,1471
2,catching_confirmed_repository_inventory.csv,.csv,reports/tables/catching_confirmed_repository_inventory.csv,6755
3,catching_confirmed_repository_registry.csv,.csv,reports/tables/catching_confirmed_repository_registry.csv,1317
4,catching_direct_validation_limitation.csv,.csv,reports/tables/catching_direct_validation_limitation.csv,248
5,catching_direct_validation_registry.csv,.csv,data/public_v2/interim/catching_direct_validation_registry.csv,928
6,catching_field_policy.csv,.csv,reports/tables/catching_field_policy.csv,645
7,catching_final_usage_decision.csv,.csv,reports/tables/catching_final_usage_decision.csv,474
8,catching_labels_consensus_long.csv,.csv,data/public_v2/interim/catching_labels_consensus_long.csv,4768
9,catching_labels_evaluator.csv,.csv,data/public_v2/interim/catching_labels_evaluator.csv,14818


## 11. Artefactos prioritarios esperados

Estos son outputs que deberían existir por las fases anteriores. Que alguno falte no implica inventarlo: se reportará y se resolverá en el notebook 25.


In [9]:
expected_artifacts = pd.DataFrame([
    {
        "logical_name": "UICrit/RICO same-instance proxy dataset",
        "expected_filename": "uicrit_rico_proxy_dataset.csv",
        "importance_for_ssl": "U_CANDIDATE"
    },
    {
        "logical_name": "Catching static validation dataset",
        "expected_filename": "catching_static_validation_dataset.csv",
        "importance_for_ssl": "L_CANDIDATE"
    },
    {
        "logical_name": "29042675 recovered candidate targets",
        "expected_filename": "vt_recovered_candidate_targets.csv",
        "importance_for_ssl": "L_CANDIDATE_PENDING_X"
    },
    {
        "logical_name": "Feature compatibility matrix",
        "expected_filename": "feature_compatibility_matrix.csv",
        "importance_for_ssl": "INPUT_TO_NOTEBOOK_25"
    },
    {
        "logical_name": "Final direct-label class audit",
        "expected_filename": "direct_label_class_audit_final.csv",
        "importance_for_ssl": "LABEL_AUDIT"
    }
])

available_names = set(
    artifact_inventory["name"].tolist()
    if not artifact_inventory.empty
    else []
)

expected_artifacts["found"] = (
    expected_artifacts["expected_filename"]
    .isin(available_names)
)

display(expected_artifacts)


,logical_name,expected_filename,importance_for_ssl,found
0,UICrit/RICO same-instance proxy dataset,uicrit_rico_proxy_dataset.csv,U_CANDIDATE,True
1,Catching static validation dataset,catching_static_validation_dataset.csv,L_CANDIDATE,True
2,29042675 recovered candidate targets,vt_recovered_candidate_targets.csv,L_CANDIDATE_PENDING_X,True
3,Feature compatibility matrix,feature_compatibility_matrix.csv,INPUT_TO_NOTEBOOK_25,True
4,Final direct-label class audit,direct_label_class_audit_final.csv,LABEL_AUDIT,False


## 12. Semilla del registro de features

In [10]:
feature_dictionary_candidates = [
    REPORTS_DIR / "feature_dictionary_validation.csv",
    REPORTS_DIR / "feature_dictionary.csv"
]

feature_dictionary_path = next(
    (
        path
        for path in feature_dictionary_candidates
        if path.exists()
    ),
    None
)

if feature_dictionary_path is not None:
    feature_dictionary = pd.read_csv(feature_dictionary_path)

    if (
        "rol_ml" in feature_dictionary.columns
        and "variable" in feature_dictionary.columns
    ):
        canonical_x_seed = (
            feature_dictionary[
                feature_dictionary["rol_ml"] == "X"
            ]
            .copy()
        )
    else:
        canonical_x_seed = feature_dictionary.copy()
else:
    feature_dictionary = pd.DataFrame()
    canonical_x_seed = pd.DataFrame()

print(
    "Feature dictionary encontrado:",
    feature_dictionary_path
)

print(
    "Variables X semilla:",
    len(canonical_x_seed)
)

display(canonical_x_seed)


Feature dictionary encontrado: /Users/uriel/Documents/machine_learning/reports/tables/feature_dictionary_validation.csv
Variables X semilla: 40


,variable,categoria,rol_ml,a11y_estado,repo01_estado,delfi_estado,irr_aporte,requiere_experto,captura_extension,estado_final
4,task_time_s,TEMPORAL,X,DIRECTA_O_DERIVADA,NO,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
5,mean_response_time_ms,TEMPORAL,X,NO,NO,NO,NO,False,DERIVADA,RECOLECCION_PROPIA
6,max_response_time_ms,TEMPORAL,X,NO,NO,NO,NO,False,DERIVADA,RECOLECCION_PROPIA
7,mean_pre_action_pause_s,TEMPORAL,X,NO,DERIVABLE_REQUIERE_VALIDACION,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
8,click_count,INTERACCION,X,DIRECTA_O_DERIVADA,DERIVABLE_VALIDADA_EN_MUESTRA,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
9,keypress_count,INTERACCION,X,DIRECTA_O_DERIVADA,VALIDADA,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
10,scroll_count,INTERACCION,X,DIRECTA_O_DERIVADA,DERIVABLE,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
11,backtracking_count,INTERACCION,X,DIRECTA_O_DERIVADA,NO,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
12,repeated_action_count,INTERACCION,X,DIRECTA_O_DERIVADA,DERIVABLE_CON_REGLA_EXPERIMENTAL,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
13,navigation_steps,INTERACCION,X,DIRECTA_O_DERIVADA,PARCIAL,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS


La lista anterior es una **referencia canónica de la extensión**, no el espacio de entrenamiento final.

El notebook 25 deberá determinar:

- qué variables tienen equivalencia empírica real;
- cuáles son solo construct proxies;
- cuáles no existen en las fuentes públicas;
- y cuál es el `X_common` que sí puede utilizar el modelo semisupervisado.


## 13. Contrato de salida del notebook 25

In [11]:
notebook25_contract = pd.DataFrame([
    {
        "output": "semisupervised_feature_mapping.csv",
        "purpose": "Mapeo fuente-específico a features candidatas comunes."
    },
    {
        "output": "semisupervised_feature_space.csv",
        "purpose": "Lista definitiva de X_common con definición, tipo y semántica."
    },
    {
        "output": "semisupervised_source_feature_coverage.csv",
        "purpose": "Cobertura de X_common por fuente."
    },
    {
        "output": "semisupervised_row_eligibility_by_source.csv",
        "purpose": "Decidir qué fuentes pueden entrar a L/U después del feature audit."
    }
])

display(notebook25_contract)


,output,purpose
0,semisupervised_feature_mapping.csv,Mapeo fuente-específico a features candidatas comunes.
1,semisupervised_feature_space.csv,"Lista definitiva de X_common con definición, tipo y semántica."
2,semisupervised_source_feature_coverage.csv,Cobertura de X_common por fuente.
3,semisupervised_row_eligibility_by_source.csv,Decidir qué fuentes pueden entrar a L/U después del feature audit.


## 14. Decisión metodológica del notebook 24

In [12]:
design_decision = pd.DataFrame([
    {
        "decision": "USE_SEMISUPERVISED_FRAMEWORK",
        "value": True,
        "reason": "Los datasets públicos presentan etiquetas Nielsen parciales y múltiples fuentes de X no etiquetada."
    },
    {
        "decision": "ASSEMBLE_MASTER_ROWS_NOW",
        "value": False,
        "reason": "No se concatenarán filas hasta definir un espacio X compatible en notebook 25."
    },
    {
        "decision": "MODEL_TARGETS_SEPARATELY",
        "value": True,
        "reason": "Cada heurística puede tener distinta disponibilidad de Y por instancia."
    },
    {
        "decision": "ALLOW_PROXY_AS_DIRECT_Y",
        "value": False,
        "reason": "La procedencia proxy se conserva para experimentos separados."
    },
    {
        "decision": "PSEUDO_LABELS_ARE_GROUND_TRUTH",
        "value": False,
        "reason": "Las pseudo-etiquetas solo sirven para entrenamiento SSL, nunca para evaluación."
    },
    {
        "decision": "EXTENSION_DEVELOPMENT_REQUIRED_NOW",
        "value": False,
        "reason": "Primero debe demostrarse que existe un X_common entrenable con los datasets públicos disponibles."
    }
])

display(design_decision)


,decision,value,reason
0,USE_SEMISUPERVISED_FRAMEWORK,True,Los datasets públicos presentan etiquetas Nielsen parciales y múltiples fuentes de X no etiquetada.
1,ASSEMBLE_MASTER_ROWS_NOW,False,No se concatenarán filas hasta definir un espacio X compatible en notebook 25.
2,MODEL_TARGETS_SEPARATELY,True,Cada heurística puede tener distinta disponibilidad de Y por instancia.
3,ALLOW_PROXY_AS_DIRECT_Y,False,La procedencia proxy se conserva para experimentos separados.
4,PSEUDO_LABELS_ARE_GROUND_TRUTH,False,"Las pseudo-etiquetas solo sirven para entrenamiento SSL, nunca para evaluación."
5,EXTENSION_DEVELOPMENT_REQUIRED_NOW,False,Primero debe demostrarse que existe un X_common entrenable con los datasets públicos disponibles.


## 15. Guardar artefactos

In [13]:
source_registry.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_source_registry.csv",
    index=False
)

label_provenance_registry.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_label_provenance_registry.csv",
    index=False
)

master_schema.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_master_schema.csv",
    index=False
)

integration_rules.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_integration_rules.csv",
    index=False
)

artifact_inventory.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_input_artifact_inventory.csv",
    index=False
)

expected_artifacts.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_expected_artifacts.csv",
    index=False
)

canonical_x_seed.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_canonical_x_seed.csv",
    index=False
)

notebook25_contract.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_notebook25_contract.csv",
    index=False
)

design_decision.to_csv(
    SEMISUP_REPORTS_DIR / "semisupervised_master_design_decision.csv",
    index=False
)


In [14]:
expected_outputs = [
    "semisupervised_source_registry.csv",
    "semisupervised_label_provenance_registry.csv",
    "semisupervised_master_schema.csv",
    "semisupervised_integration_rules.csv",
    "semisupervised_input_artifact_inventory.csv",
    "semisupervised_expected_artifacts.csv",
    "semisupervised_canonical_x_seed.csv",
    "semisupervised_notebook25_contract.csv",
    "semisupervised_master_design_decision.csv"
]

for filename in expected_outputs:
    path = SEMISUP_REPORTS_DIR / filename

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


semisupervised_source_registry.csv ✅
semisupervised_label_provenance_registry.csv ✅
semisupervised_master_schema.csv ✅
semisupervised_integration_rules.csv ✅
semisupervised_input_artifact_inventory.csv ✅
semisupervised_expected_artifacts.csv ✅
semisupervised_canonical_x_seed.csv ✅
semisupervised_notebook25_contract.csv ✅
semisupervised_master_design_decision.csv ✅


## Punto de detención

El notebook 24 queda cerrado cuando:

- no hay errores de ejecución;
- se genera el registro de fuentes;
- se genera el esquema maestro;
- se genera el registro de procedencia de etiquetas;
- se inventan **cero** labels;
- no se concatenan aún datasets incompatibles;
- se obtiene el inventario real de artefactos disponibles.

El siguiente archivo será:

`25_semisupervised_feature_space.ipynb`

y su tarea central será responder:

> **¿Qué variables X pueden medirse de forma suficientemente compatible en los datasets públicos y posteriormente en la extensión web?**

Solo después de esa respuesta se construirá `L ∪ U`.
